# 混合精度：bit allocation 三步框架

对应文章：《大模型量化算法（25）：混合精度》
https://lrypcy.github.io/2026/09/19/llm-quant-25-mixed-precision/

| 实验 | 要回答的问题 |
|---|---|
| A | 敏感度为什么是长尾的？"统一比特"到底亏在哪 |
| B | 三种 bit allocation 求解器对比：边际收益贪心 / 拉格朗日扫描 / 精确 DP |
| C | 拉格朗日的 λ 扫描给出完整 Pareto 前沿，与精确解差多少？ |
| D | 敏感度度量选哪个？Hessian / 激活幅度 / 重建误差 三种度量的排序一致性 |
| E | MoE 专项：路由动力学 vs 调用频率，"高频专家更重要"是错的 |
| F | 硬件可行性校验：显存下降 vs 混合精度额外开销，净收益是否成立 |

纯 numpy 合成问题（L 个可分配单元），SEED=0，CPU 秒级。

In [1]:
import os, json, collections
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"
CFG = {
    "smoke": dict(L=32, lam_grid=500, bit_budget_ratio=0.70, n_experts=16, n_moe_tok=4000),
    "full":  dict(L=64, lam_grid=1500, bit_budget_ratio=0.70, n_experts=32, n_moe_tok=20000),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
B_BITS = [2, 3, 4, 6, 8]
IDX = {b: i for i, b in enumerate(B_BITS)}
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'L': 32, 'lam_grid': 500, 'bit_budget_ratio': 0.7, 'n_experts': 16, 'n_moe_tok': 4000}


## A · 敏感度长尾：统一比特为什么是坏主意

合成 $L$ 个可独立分配 bit 的单元（层 / 矩阵 / 专家）。每个单元 $\ell$ 有参数量 $n_\ell$、
敏感度 $\Lambda_\ell$、误差随 bit 的下降速率 $\alpha_\ell$：

$$\mathcal{E}_\ell(b)=\frac{\Lambda_\ell}{(2^b-1)^{\alpha_\ell}},\qquad \mathcal{S}_\ell(b)=\frac{n_\ell\, b}{8}\ \text{bytes}$$

敏感度用 **Pareto 分布**（长尾）采样并降序排列——这是真实 LLM 上反复观测到的形态：
少数层贡献了大部分量化损失。先看这个长尾有多陡。

In [2]:
rng = np.random.default_rng(SEED)
L = CFG["L"]
sizes = rng.integers(20, 200, size=L).astype(float)          # 每单元参数量
sens_full = np.sort(rng.pareto(1.6, size=L) + 0.05)[::-1]    # 长尾敏感度（降序）
alpha = rng.uniform(0.8, 1.6, size=L)                        # 误差下降速率因单元而异

E = np.array([[sens_full[l] / ((2 ** b - 1) ** alpha[l]) for b in B_BITS] for l in range(L)])
S = np.array([[sizes[l] * b / 8 for b in B_BITS] for l in range(L)])

budget = CFG["bit_budget_ratio"] * S[:, IDX[4]].sum()
uni_candidates = [b for b in B_BITS if S[:, IDX[b]].sum() <= budget]
uni_b = max(uni_candidates)
base_err = E[:, IDX[uni_b]].sum()

log("=== A 敏感度长尾 ===")
log(f"单元数 L={L}   总参数={sizes.sum():.0f}")
log(f"预算 = 统一 4-bit 体积的 {CFG['bit_budget_ratio']:.0%} = {budget:.0f} B")
for b in B_BITS:
    v = S[:, IDX[b]].sum()
    log(f"  统一 {b}-bit: 体积={v:8.0f}B  误差={E[:, IDX[b]].sum():9.4f}  "
        f"{'<= 预算 OK' if v <= budget else '超预算'}")
log(f"-> 预算内能塞下的最大统一 bit = {uni_b}-bit，误差基线 base_err = {base_err:.4f}")
for k in (1, 2, 4, 8):
    log(f"  top-{k} 单元占敏感度总和的 {sens_full[:k].sum()/sens_full.sum():.1%}")
log(f"  敏感度 max/min = {sens_full.max()/sens_full.min():.1f}x  (长尾)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(np.arange(L), sens_full / sens_full.sum())
ax[0].set_title("per-unit sensitivity (Pareto, sorted)")
ax[0].set_xlabel("unit rank (desc)"); ax[0].set_ylabel("share of total sensitivity")
ax[0].axhline(1.0 / L, color="crimson", ls="--", lw=1, label=f"uniform share 1/L={1/L:.3f}")
ax[0].legend(fontsize=8)
cum = np.cumsum(sens_full) / sens_full.sum()
ax[1].plot(np.arange(1, L + 1), cum, marker="o", ms=3)
ax[1].set_title("cumulative sensitivity share")
ax[1].set_xlabel("top-k units"); ax[1].set_ylabel("cumulative share")
ax[1].grid(alpha=0.3)
savefig(fig, "mp_a_sensitivity_tail.png")

=== A 敏感度长尾 ===
单元数 L=32   总参数=3658
预算 = 统一 4-bit 体积的 70% = 1280 B
  统一 2-bit: 体积=     914B  误差=  10.6656  <= 预算 OK
  统一 3-bit: 体积=    1372B  误差=   4.2521  超预算
  统一 4-bit: 体积=    1829B  误差=   1.9083  超预算
  统一 6-bit: 体积=    2744B  误差=   0.4454  超预算
  统一 8-bit: 体积=    3658B  误差=   0.1135  超预算
-> 预算内能塞下的最大统一 bit = 2-bit，误差基线 base_err = 10.6656
  top-1 单元占敏感度总和的 16.8%
  top-2 单元占敏感度总和的 31.0%
  top-4 单元占敏感度总和的 43.3%
  top-8 单元占敏感度总和的 63.4%
  敏感度 max/min = 84.6x  (长尾)


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_a_sensitivity_tail.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_a_sensitivity_tail.png'

## B · 三种求解器：贪心 / 拉格朗日 / 精确 DP

形式化是**多重选择背包问题**（NP-hard，但规模小可精确解）：

$$\min_{b_1..b_L}\sum_\ell \mathcal{E}_\ell(b_\ell)\quad\text{s.t.}\quad\sum_\ell \mathcal{S}_\ell(b_\ell)\le B$$

- **边际收益贪心**：全部从 2-bit 起，每步挑「单位体积误差下降最大」的单元提一档
- **拉格朗日扫描**（工业界默认）：$\min_\ell[\mathcal{E}_\ell(b)+\lambda\mathcal{S}_\ell(b)]$，目标可分离 → 逐单元独立求解，扫 $\lambda$ 得 Pareto 前沿
- **精确 DP**：多重选择背包动态规划，作为最优性参照

对照组是**预算内最好的统一方案**（这里只有 2-bit 能塞进去）——这才是混合精度真正的对手，
不是「同样 bit 的统一方案」。

In [3]:
def greedy(E, S, budget):
    L = E.shape[0]
    b = np.zeros(L, dtype=int)
    vol = S[np.arange(L), b].sum()
    while True:
        gains = []
        for l in range(L):
            if b[l] + 1 < len(B_BITS):
                dv = S[l, b[l] + 1] - S[l, b[l]]
                de = E[l, b[l]] - E[l, b[l] + 1]
                if vol + dv <= budget and de > 0:
                    gains.append((de / dv, l, dv))
        if not gains:
            break
        gains.sort(reverse=True)
        _, l, dv = gains[0]
        b[l] += 1; vol += dv
    return b, float(E[np.arange(L), b].sum()), float(vol)


def lagrangian(E, S, budget, grid=None):
    if grid is None:
        grid = np.logspace(-5, 2, CFG["lam_grid"])
    best = (None, None, None, None)
    for lam in grid:
        j = np.argmin(E + lam * S, axis=1)
        v = S[np.arange(E.shape[0]), j].sum()
        if v <= budget:
            e = float(E[np.arange(E.shape[0]), j].sum())
            if best[0] is None or e < best[0]:
                best = (e, float(lam), j, float(v))
    return best  # err, lam, idx, vol


def exact_dp(E, S, budget, scale=8):
    """多重选择背包精确 DP。不可行时返回 (None, inf, nan)。"""
    L = E.shape[0]
    b_i = int(budget * scale)
    if b_i < (S[:, 0] * scale).astype(int).sum():
        return None, float("inf"), float("nan")     # 预算连全最低 bit 都装不下
    S_i = (S * scale).astype(int)
    dp = np.full(b_i + 1, np.inf); dp[0] = 0.0
    keep = np.zeros((L, b_i + 1), dtype=int)
    for l in range(L):
        ndp = np.full(b_i + 1, np.inf); nk = np.zeros(b_i + 1, dtype=int)
        for bi in range(len(B_BITS)):
            s_, e_ = S_i[l, bi], E[l, bi]
            if s_ > b_i:
                continue
            cand = np.roll(dp, s_) + e_
            cand[:s_] = np.inf
            better = cand < ndp
            ndp[better] = cand[better]; nk[better] = bi
        dp = ndp; keep[l] = nk
    v = int(np.argmin(dp))
    if not np.isfinite(dp[v]):
        return None, float("inf"), float("nan")
    assign = np.zeros(L, dtype=int); vv = v
    for l in range(L - 1, -1, -1):
        assign[l] = keep[l, vv]; vv -= S_i[l, assign[l]]
    return assign, float(dp[v]), v / scale


g_idx, g_err, g_vol = greedy(E, S, budget)
l_err, l_lam, l_idx, l_vol = lagrangian(E, S, budget)
d_idx, d_err, d_vol = exact_dp(E, S, budget)

log("=== B 三种求解器 vs 统一基线 ===")
log(f"{'方法':<22}{'体积(B)':>10}{'误差':>12}{'相对基线':>12}{'预算内':>8}")
rows = [("统一 %d-bit 基线" % uni_b, S[:, IDX[uni_b]].sum(), base_err),
        ("边际收益贪心", g_vol, g_err),
        ("拉格朗日扫描", l_vol, l_err),
        ("精确 DP", d_vol, d_err)]
for name, v, e in rows:
    log(f"{name:<22}{v:>10.0f}{e:>12.4f}{e/base_err:>12.3f}{'是' if v <= budget+1e-9 else '否':>8}")
log(f"拉格朗日 lambda* = {l_lam:.6f}   与精确 DP 的相对差距 = {(l_err-d_err)/d_err:+.2%}")
log(f"贪心 vs 精确 DP 差距           = {(g_err-d_err)/d_err:+.2%}")
log(f"bit 分布  贪心   : {dict(collections.Counter([B_BITS[i] for i in g_idx]))}")
log(f"bit 分布  拉格朗日: {dict(collections.Counter([B_BITS[i] for i in l_idx]))}")
log(f"bit 分布  精确 DP : {dict(collections.Counter([B_BITS[i] for i in d_idx]))}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for a, (name, idx) in zip(ax, [("greedy", g_idx), ("lagrangian", l_idx), ("exact DP", d_idx)]):
    a.bar(np.arange(L), [B_BITS[i] for i in idx], color="steelblue")
    a.axhline(uni_b, color="crimson", ls="--", lw=1.2, label=f"uniform {uni_b}-bit baseline")
    a.set_title(name); a.set_xlabel("unit"); a.set_ylim(0, 9)
    a.legend(fontsize=8)
ax[0].set_ylabel("assigned bits")
savefig(fig, "mp_b_bit_allocation.png")

=== B 三种求解器 vs 统一基线 ===
方法                         体积(B)          误差        相对基线     预算内
统一 2-bit 基线                  914     10.6656       1.000       是
边际收益贪心                      1280      2.8871       0.271       是
拉格朗日扫描                      1274      2.9155       0.273       是
精确 DP                       1280      2.8852       0.271       是
拉格朗日 lambda* = 0.006392   与精确 DP 的相对差距 = +1.05%
贪心 vs 精确 DP 差距           = +0.07%
bit 分布  贪心   : {4: 2, 6: 6, 3: 10, 2: 14}
bit 分布  拉格朗日: {4: 3, 6: 5, 3: 10, 2: 14}
bit 分布  精确 DP : {4: 4, 6: 4, 3: 11, 2: 13}


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_b_bit_allocation.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_b_bit_allocation.png'

## C · λ 扫描 = 完整的精度-体积权衡曲线

拉格朗日方法真正的价值不在于它有多准，而在于**扫一遍 λ 就白拿了整条 Pareto 前沿**：
每个 λ 给出一个（体积, 误差）点，挑预算内误差最小的那个即可。这里把这条前沿画出来，
并叠加精确 DP 的可达点做校验。

In [4]:
lam_grid = np.logspace(-5, 2, CFG["lam_grid"])
pts = []
for lam in lam_grid:
    j = np.argmin(E + lam * S, axis=1)
    pts.append((S[np.arange(L), j].sum(), E[np.arange(L), j].sum(), lam))
pts = np.array(pts)
order = np.argsort(pts[:, 0])
pv, pe = pts[order, 0], pts[order, 1]
# 取下包络（Pareto 前沿）
env_v, env_e = [], []
best = np.inf
for v, e in zip(pv, pe):
    if e < best - 1e-15:
        env_v.append(v); env_e.append(e); best = e

min_ratio = S[:, IDX[B_BITS[0]]].sum() / S[:, IDX[4]].sum()     # 全最低 bit 也要占的体积比
log(f"最低可行预算比 = {min_ratio:.3f}（全部单元给最低 bit={B_BITS[0]}）")
dp_pts = []
for ratio in np.linspace(min_ratio + 0.02, 1.0, 16):
    bgt = ratio * S[:, IDX[4]].sum()
    _, e_dp, v_dp = exact_dp(E, S, bgt)
    if np.isfinite(e_dp):
        dp_pts.append((v_dp, e_dp))
dp_pts = np.array(dp_pts)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(pv, pe, s=6, alpha=0.35, label="all lambda points")
ax[0].plot(env_v, env_e, color="darkorange", lw=2, label="Pareto envelope")
ax[0].scatter(dp_pts[:, 0], dp_pts[:, 1], s=28, marker="x", color="green", label="exact DP")
ax[0].axvline(budget, color="crimson", ls="--", lw=1.2, label=f"budget={budget:.0f}B")
ax[0].scatter([S[:, IDX[b]].sum() for b in B_BITS], [E[:, IDX[b]].sum() for b in B_BITS],
              s=45, marker="s", color="black", label="uniform b-bit")
ax[0].set_xlabel("total bytes"); ax[0].set_ylabel("total error")
ax[0].set_title("accuracy-size trade-off frontier"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
ax[1].plot(np.arange(1, len(env_e) + 1), env_e / base_err, marker="o", ms=3)
ax[1].axhline(1.0, color="crimson", ls="--", lw=1, label="uniform 2-bit baseline")
ax[1].set_xlabel("Pareto point index (size ascending)"); ax[1].set_ylabel("error / baseline")
ax[1].set_title("how much mixed-precision buys you"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
savefig(fig, "mp_c_pareto_frontier.png")
log("=== C Pareto 前沿 ===")
log(f"前沿点数={len(env_e)}  预算点误差/基线 = {l_err/base_err:.3f}")
_e100 = exact_dp(E, S, S[:, IDX[4]].sum())[1]
log(f"若预算放宽到统一 4-bit 的 100%，混合精度误差 = {_e100:.4f}（{_e100/base_err:.4f} x 基线，"
    f"相对统一 4-bit 的 {E[:, IDX[4]].sum():.4f} 又低 {(1-_e100/E[:, IDX[4]].sum())*100:.1f}%）")

最低可行预算比 = 0.500（全部单元给最低 bit=2）


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_c_pareto_frontier.png
=== C Pareto 前沿 ===
前沿点数=107  预算点误差/基线 = 0.273
若预算放宽到统一 4-bit 的 100%，混合精度误差 = 0.9489（0.0890 x 基线，相对统一 4-bit 的 1.9083 又低 50.3%）


## D · 敏感度度量选哪个？

三种主流度量：

| 度量 | 代表 | 公式 |
|---|---|---|
| 二阶 / Hessian 类 | OBQ / GPTQ 系 | $\Lambda_\ell \propto \mathrm{diag}(H_\ell)$，或 $\|W_\ell\| \cdot \mathrm{diag}(H)$ |
| 激活幅度类 | AWQ / OWQ | $\Lambda_\ell \propto \|\mathrm{diag}(X_\ell)\|$（激活 scale 大 → 权重误差被放大） |
| 重建误差类 | 最朴素 | 直接量 $\mathcal{E}_\ell(b_{\min})$ |

问题：它们给出的**排序**一致吗？排序不一致 → 选错度量会直接把 bit 分错层。
这里用真实可算的 Hessian（$H=X^\top X$）和激活幅度分别给每个单元打分，
比较 Spearman 秩相关，并看「用错度量的贪心」最终误差恶化多少。

In [5]:
def spearman(a, b):
    ra = np.argsort(np.argsort(a)).astype(float)
    rb = np.argsort(np.argsort(b)).astype(float)
    ra -= ra.mean(); rb -= rb.mean()
    return float(ra @ rb / np.sqrt((ra @ ra) * (rb @ rb) + 1e-30))


# ---- 换一个「实测」探针：每个单元的量化代价是量出来的，不是公式造的 ----
# 难点来源有两个，且互相独立：激活的离群通道强度、权重自身的重尾程度。
Lp, d_in, d_out, n_cal = 32, 16, 16, 256
r2 = np.random.default_rng(SEED + 1)
Xs_p, Ws_p, Hs_p = [], [], []
for _ in range(Lp):
    Xp = r2.normal(0, 1, (n_cal, d_in))
    n_o = int(r2.integers(0, 4))                          # 离群通道数 0~3
    if n_o:
        cols = r2.choice(d_in, n_o, replace=False)
        Xp[:, cols] *= float(np.exp(r2.uniform(0.8, 3.2)))   # 离群倍率
    Wp = r2.normal(0, 1, (d_in, d_out))
    if r2.random() < 0.5:                                  # 一半单元权重带重尾
        msk = r2.random(Wp.shape) < float(r2.uniform(0.02, 0.15))
        Wp[msk] *= float(np.exp(r2.uniform(0.8, 2.5)))
    Xs_p.append(Xp); Ws_p.append(Wp); Hs_p.append(Xp.T @ Xp / n_cal)

def measured_err(Xp, Wp, b):
    """实测：激活 per-tensor 8-bit + 权重 per-tensor b-bit 后的输出相对误差"""
    sa = np.abs(Xp).max() / (2 ** 7 - 1)
    Xq = np.round(Xp / sa) * sa
    if b >= 30:
        Wq = Wp
    else:
        # per-tensor 权重量化：离群值会劫持整个 scale，单元间差异才会显现
        sw = np.abs(Wp).max() / (2 ** (b - 1) - 1)
        Wq = np.round(Wp / sw) * sw
    ref = Xp @ Wp
    return float(np.linalg.norm(ref - Xq @ Wq) / np.linalg.norm(ref))

# 真值：分配位宽(2-bit)上的实测代价；度量：各候选指标
TARGET_B, REF_B = 2, 3
true_err = np.array([measured_err(Xs_p[l], Ws_p[l], TARGET_B) for l in range(Lp)])
metric_recon = np.array([measured_err(Xs_p[l], Ws_p[l], REF_B) for l in range(Lp)])
metric_hess = np.array([np.sqrt(np.abs(np.diag(H)).mean()) * np.abs(W).mean()
                        for H, W in zip(Hs_p, Ws_p)])
metric_act = np.array([np.abs(X).mean() * np.abs(W).mean() for X, W in zip(Xs_p, Ws_p)])
metric_wmag = np.array([np.abs(W).mean() for W in Ws_p])

log("=== D 敏感度度量：谁能预测真实的量化代价 ===")
log(f"（实测探针：{Lp} 个单元；真值 = W{TARGET_B}A8 的输出相对误差；"
    f"重建误差类度量 = W{REF_B}A8 的实测误差）")
met = {"Hessian 类": metric_hess, "激活幅度类": metric_act,
       "权重幅度": metric_wmag, f"重建误差类(W{REF_B} 实测)": metric_recon}
rho_true = {}
log(f"{'度量':<22}{'Spearman vs 真值':>18}")
for nm, m in met.items():
    rho_true[nm] = spearman(m, true_err)
    log(f"{nm:<22}{rho_true[nm]:>18.3f}")
best_metric = max(rho_true, key=lambda k: rho_true[k])
log("-" * 78)
log(f"  读数：{best_metric} 与真值最相关（rho={rho_true[best_metric]:+.3f}）。")
log("        Hessian / 激活幅度刻画的是『误差被放大的倍率』，但一个单元量化代价高不高")
log("        还取决于它自己难不难（离群通道、权重重尾）——这两件事与倍率并不完全相关。")
log("        => 文章 §3.4 的判断：重建误差类『最朴素但最可靠』，因为它直接量你要优化的东西。")

# ---- 在实测探针上端到端跑一遍：误差表来自实测，而不是公式 ----
E_p = np.array([[measured_err(Xs_p[l], Ws_p[l], b) for b in B_BITS] for l in range(Lp)])
sizes_p = np.array([Ws_p[l].size for l in range(Lp)], dtype=float)
S_p = np.array([[sizes_p[l] * b / 8 for b in B_BITS] for l in range(Lp)])
bud_p = 0.7 * S_p[:, IDX[4]].sum()
uni_p = max([b for b in B_BITS if S_p[:, IDX[b]].sum() <= bud_p])
base_p = E_p[:, IDX[uni_p]].sum()

def greedy_with_metric(metric, Et, St, bud):
    order = np.argsort(-metric); Lc = Et.shape[0]
    b = np.zeros(Lc, dtype=int); vol = St[np.arange(Lc), b].sum()
    improved = True
    while improved:
        improved = False
        for l in order:
            if b[l] + 1 < len(B_BITS):
                dv = St[l, b[l] + 1] - St[l, b[l]]
                de = Et[l, b[l]] - Et[l, b[l] + 1]
                if vol + dv <= bud and de > 0:
                    b[l] += 1; vol += dv; improved = True
    return float(Et[np.arange(Lc), b].sum())

gp_i, gp_err, _ = greedy(E_p, S_p, bud_p)
lp_err, _, lp_idx, _ = lagrangian(E_p, S_p, bud_p)
dp_err_p = exact_dp(E_p, S_p, bud_p)[1]
log(f"\n实测探针上端到端（预算 = 统一 4-bit 的 70%，统一 {uni_p}-bit 基线 = {base_p:.4f}）:")
log(f"  边际收益贪心  : {gp_err:.4f}  ({gp_err/base_p:.3f} x 基线)")
log(f"  拉格朗日扫描  : {lp_err:.4f}  ({lp_err/base_p:.3f} x 基线)")
log(f"  精确 DP       : {dp_err_p:.4f}  ({dp_err_p/base_p:.3f} x 基线)")
err_by_metric = {}
for nm, m in met.items():
    err_by_metric[nm] = greedy_with_metric(m, E_p, S_p, bud_p)
    log(f"  用 {nm:<22} 排序驱动贪心 -> {err_by_metric[nm]:.4f}  "
        f"({err_by_metric[nm]/base_p:.3f} x 基线)")
log("-> 排序型贪心假设『误差下降速率各单元相同』，实测数据上这一假设不成立；")
log("   只有在拿到逐单元误差表 E_l(b) 之后，贪心/拉格朗日才解得准。")

fig, ax = plt.subplots(1, 5, figsize=(18, 3.4))
for a, (nm, m) in zip(ax, list(met.items())):
    a.scatter(m, true_err, s=22)
    a.set_xlabel(nm, fontsize=8); a.set_ylabel(f"true err @W{TARGET_B}", fontsize=8)
    a.set_title(f"rho={rho_true[nm]:+.3f}", fontsize=9); a.grid(alpha=0.3)
ax[4].bar(list(rho_true.keys()), list(rho_true.values()),
          color=["#4c72b0", "#dd8452", "#8c8c8c", "#55a868"])
ax[4].set_ylabel("Spearman vs measured cost", fontsize=8)
ax[4].set_title("recon-type metric wins", fontsize=9)
ax[4].tick_params(axis="x", labelsize=6, rotation=30); ax[4].grid(alpha=0.3, axis="y")
savefig(fig, "mp_d_metric_agreement.png")

=== D 敏感度度量：谁能预测真实的量化代价 ===
（实测探针：32 个单元；真值 = W2A8 的输出相对误差；重建误差类度量 = W3A8 的实测误差）
度量                        Spearman vs 真值
Hessian 类                          0.051
激活幅度类                              0.018
权重幅度                              -0.096
重建误差类(W3 实测)                       0.424
------------------------------------------------------------------------------
  读数：重建误差类(W3 实测) 与真值最相关（rho=+0.424）。
        Hessian / 激活幅度刻画的是『误差被放大的倍率』，但一个单元量化代价高不高
        还取决于它自己难不难（离群通道、权重重尾）——这两件事与倍率并不完全相关。
        => 文章 §3.4 的判断：重建误差类『最朴素但最可靠』，因为它直接量你要优化的东西。

实测探针上端到端（预算 = 统一 4-bit 的 70%，统一 2-bit 基线 = 25.8813）:
  边际收益贪心  : 14.4492  (0.558 x 基线)
  拉格朗日扫描  : 14.4278  (0.557 x 基线)
  精确 DP       : 14.4278  (0.557 x 基线)
  用 Hessian 类              排序驱动贪心 -> 16.7152  (0.646 x 基线)
  用 激活幅度类                  排序驱动贪心 -> 16.7152  (0.646 x 基线)
  用 权重幅度                   排序驱动贪心 -> 17.0974  (0.661 x 基线)
  用 重建误差类(W3 实测)           排序驱动贪心 -> 16.7218  (0.646 x 基线)
-> 排序型贪心假设『误差下降速率各单元相同』，实测数据上这一假设不成立；
   只有在拿到逐单元误差表

/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_72420/1089772198.py:18: UserWarning: Glyph 31867 (\N{CJK UNIFIED IDEOGRAPH-7C7B}) missing from font(s) DejaVu Sans.
  p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_72420/1089772198.py:18: UserWarning: Glyph 28608 (\N{CJK UNIFIED IDEOGRAPH-6FC0}) missing from font(s) DejaVu Sans.
  p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_72420/1089772198.py:18: UserWarning: Glyph 27963 (\N{CJK UNIFIED IDEOGRAPH-6D3B}) missing from font(s) DejaVu Sans.
  p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_72420/1089772198.py:18: UserWarning: Glyph 24133 (\N{CJK UNIFIED IDEOGRAPH-5E45}) missing from font(s) DejaVu Sans.
  p = os.path.join(RES, 

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_d_metric_agreement.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_d_metric_agreement.png'

## E · MoE 专项：路由动力学 vs 调用频率

直觉是「调用频率高的专家更重要，该给高精度」——**这是错的**。MoE 量化真正脆弱的是
**router 权重的 $\ell_2$ 范数变化最小的那些专家**：router 权重一扰动，它们的 logit 变化
相对自身量级最大，token 路由就被改道了。而这个指标与调用频率**基本不相关**。

In [6]:
n_e = CFG["n_experts"]
r3 = np.random.default_rng(SEED + 2)
router_norm_change = r3.exponential(1.0, size=n_e)     # 越小 -> 越脆弱
freq = r3.zipf(1.3, size=n_e).astype(float); freq /= freq.sum()
fragile = np.argsort(router_norm_change)[:max(2, n_e // 4)]
rank_of = {int(e): int(r) for r, e in enumerate(np.argsort(-freq))}

# 多轮 bootstrap：验证「脆弱度」与「调用频率」在统计上无关
rhos = []
r4 = np.random.default_rng(SEED + 3)
for _ in range(300):
    nc = r4.exponential(1.0, size=n_e)
    fr = r4.zipf(1.3, size=n_e).astype(float); fr /= fr.sum()
    rhos.append(spearman(-nc, fr))
rhos = np.array(rhos)

log("=== E MoE 路由动力学 vs 频率 ===")
log(f"专家数={n_e}   最脆弱(top {len(fragile)})专家 = {sorted(fragile.tolist())}")
log(f"它们的调用频率排名(0=最高频) = {[rank_of[int(e)] for e in fragile]}")
log(f"单次样本 Spearman(脆弱度, 频率) = {spearman(-router_norm_change, freq):+.3f}")
log(f"300 轮 bootstrap: 均值 {rhos.mean():+.3f}  std {rhos.std():.3f}  "
    f"95% 区间 [{np.percentile(rhos,2.5):+.3f}, {np.percentile(rhos,97.5):+.3f}]")
log(f"脆弱专家的频率排名散布在 {min(rank_of[int(e)] for e in fragile)} ~ "
    f"{max(rank_of[int(e)] for e in fragile)}（全区间 0~{n_e-1}）")
log("  -> rho 的置信区间覆盖 0：脆弱度与调用频率在统计上无关。")
log("     『高频专家更重要』的诊断因此是错的——高频只说明它常被用到，不说明它对扰动敏感。")

# 量化 router 后路由改道率的实证
d_model = 32
W_router = r3.normal(0, 1 / np.sqrt(d_model), (n_e, d_model))
tok = r3.normal(0, 1, (CFG["n_moe_tok"], d_model))
topk = 2
def route(W, b=8):
    s = np.abs(W).max() / (2 ** (b - 1) - 1)
    Wq = np.round(W / s) * s
    lg = tok @ Wq.T
    return np.argsort(-lg, axis=1)[:, :topk]

r_fp = route(W_router, b=32)
flip_rates = []
for b in [8, 6, 4, 3, 2]:
    r_q = route(W_router, b=b)
    flip = np.mean([len(set(r_fp[i]) ^ set(r_q[i])) > 0 for i in range(len(tok))])
    flip_rates.append(flip)
    log(f"  router W{b}-bit: token 路由改道率 = {flip:.3%}")
log("-> router 是『小矩阵 + 高杠杆』的典型：把 router 单独留在 8/16-bit 是性价比最高的一步")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(router_norm_change, freq * n_e, s=40)
for e in fragile:
    ax[0].annotate(str(e), (router_norm_change[e], freq[e] * n_e), fontsize=8, color="crimson")
ax[0].set_xlabel("router weight L2 change (small = fragile)")
ax[0].set_ylabel("call frequency (normalized)")
ax[0].set_title("fragility is NOT frequency"); ax[0].grid(alpha=0.3)
ax[1].plot([8, 6, 4, 3, 2], flip_rates, marker="o")
ax[1].set_xlabel("router weight bits"); ax[1].set_ylabel("token reroute rate")
ax[1].set_title("small matrix, high leverage"); ax[1].grid(alpha=0.3); ax[1].invert_xaxis()
savefig(fig, "mp_e_moe_router.png")

=== E MoE 路由动力学 vs 频率 ===
专家数=16   最脆弱(top 4)专家 = [0, 1, 6, 7]
它们的调用频率排名(0=最高频) = [9, 2, 11, 0]
单次样本 Spearman(脆弱度, 频率) = +0.335
300 轮 bootstrap: 均值 +0.012  std 0.269  95% 区间 [-0.513, +0.568]
脆弱专家的频率排名散布在 0 ~ 11（全区间 0~15）
  -> rho 的置信区间覆盖 0：脆弱度与调用频率在统计上无关。
     『高频专家更重要』的诊断因此是错的——高频只说明它常被用到，不说明它对扰动敏感。
  router W8-bit: token 路由改道率 = 1.475%
  router W6-bit: token 路由改道率 = 5.500%
  router W4-bit: token 路由改道率 = 20.025%
  router W3-bit: token 路由改道率 = 41.675%
  router W2-bit: token 路由改道率 = 82.800%
-> router 是『小矩阵 + 高杠杆』的典型：把 router 单独留在 8/16-bit 是性价比最高的一步


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_e_moe_router.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_e_moe_router.png'

## F · 硬件可行性校验（最常被跳过的一步）

混合精度不是免费的：packing 低效（3/5/6-bit 不是 2 的幂）、每层不同 kernel 的分派开销、
不同的 dequant 路径。工程侧报告的混合精度额外开销约 **10%~30%**。
所以判断标准是：

$$\text{净收益} = \text{显存下降幅度} - \text{额外开销} > 0$$

另外单独看 3-bit 的 packing 低效：3-bit 需要跨元素拼接，实际占用往往不是理论值的 3/8。

In [7]:
vol_mix = S[np.arange(L), l_idx].sum()
vol_uni4 = S[:, IDX[4]].sum()
mem_drop = 1 - vol_mix / vol_uni4
log("=== F 硬件可行性校验 ===")
log(f"统一 4-bit 体积 = {vol_uni4:.0f}B   混合精度体积 = {vol_mix:.0f}B")
log(f"相对统一 4-bit 显存下降 = {mem_drop*100:.1f}%")
for overhead in (0.10, 0.20, 0.30):
    net = mem_drop - overhead
    log(f"  额外开销 {overhead:.0%} -> 净收益 {net:+.3f}  {'成立' if net > 0 else '不成立，需收紧硬件约束'}")

# packing 低效：非 2 的幂次位宽
log("packing 效率（理论 bit/8 vs 实际对齐到 32-bit word 的占用）:")
pack_rows = []
for b in B_BITS:
    per_word = 32 // b                     # 一个 32-bit word 能装几个元素
    used = per_word * b
    eff = used / 32.0
    pack_rows.append((b, per_word, eff))
    log(f"  {b}-bit: 每 word 装 {per_word:2d} 个，占用 {used:2d}/32 bit，packing 效率 {eff:.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
labels = ["greedy", "lagrangian", "exactDP"]
errs = [g_err / base_err, l_err / base_err, d_err / base_err]
ax[0].bar(labels, errs, color=["#4c72b0", "#dd8452", "#55a868"])
ax[0].axhline(1.0, color="crimson", ls="--", lw=1.2, label="uniform 2-bit baseline")
ax[0].set_ylabel("total error / baseline"); ax[0].set_title("solver comparison")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3, axis="y")
ax[1].bar([str(r[0]) for r in pack_rows], [r[2] for r in pack_rows], color="#8c8c8c")
ax[1].axhline(1.0, color="black", ls="--", lw=1)
ax[1].set_ylim(0, 1.1); ax[1].set_xlabel("bit width"); ax[1].set_ylabel("packing efficiency")
ax[1].set_title("non-power-of-2 bits waste space"); ax[1].grid(alpha=0.3, axis="y")
savefig(fig, "mp_f_hardware_check.png")

=== F 硬件可行性校验 ===
统一 4-bit 体积 = 1829B   混合精度体积 = 1274B
相对统一 4-bit 显存下降 = 30.3%
  额外开销 10% -> 净收益 +0.203  成立
  额外开销 20% -> 净收益 +0.103  成立
  额外开销 30% -> 净收益 +0.003  成立
packing 效率（理论 bit/8 vs 实际对齐到 32-bit word 的占用）:
  2-bit: 每 word 装 16 个，占用 32/32 bit，packing 效率 100.0%
  3-bit: 每 word 装 10 个，占用 30/32 bit，packing 效率 93.8%
  4-bit: 每 word 装  8 个，占用 32/32 bit，packing 效率 100.0%
  6-bit: 每 word 装  5 个，占用 30/32 bit，packing 效率 93.8%
  8-bit: 每 word 装  4 个，占用 32/32 bit，packing 效率 100.0%
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_f_hardware_check.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_f_hardware_check.png'

## 结论汇总

In [8]:
summary = {
    "meta": {"mode": MODE, "seed": SEED, "L": L, "budget_bytes": float(budget),
             "budget_def": f"{CFG['bit_budget_ratio']:.0%} of uniform-4bit size"},
    "A_tail": {"uniform_bits_in_budget": uni_b, "base_err": float(base_err),
               "top4_sens_share": float(sens_full[:4].sum() / sens_full.sum()),
               "sens_max_min_ratio": float(sens_full.max() / sens_full.min())},
    "B_solvers": {"uniform_%d" % uni_b: float(base_err), "greedy": float(g_err),
                  "lagrangian": float(l_err), "exact_dp": float(d_err),
                  "greedy_rel": float(g_err / base_err), "lagrangian_rel": float(l_err / base_err),
                  "dp_rel": float(d_err / base_err),
                  "lagrangian_vs_dp_gap_pct": float((l_err - d_err) / d_err * 100),
                  "lambda_star": float(l_lam)},
    "C_frontier": {"n_points": len(env_e),
                   "err_at_budget_rel": float(l_err / base_err)},
    "D_metrics": {"target_bit": TARGET_B, "ref_bit": REF_B,
                  "spearman_vs_true": {k: float(v) for k, v in rho_true.items()},
                  "probe_end2end": {"uniform_baseline": float(base_p), "greedy": float(gp_err),
                                    "lagrangian": float(lp_err), "exact_dp": float(dp_err_p)},
                  "greedy_err_by_metric": {k: float(v) for k, v in err_by_metric.items()}},
    "E_moe": {"n_experts": n_e, "fragile": sorted(int(x) for x in fragile),
              "fragile_freq_rank": [rank_of[int(e)] for e in fragile],
              "spearman_fragility_freq": spearman(-router_norm_change, freq),
              "reroute_rate": {str(b_): float(f) for b_, f in zip([8, 6, 4, 3, 2], flip_rates)}},
    "F_hardware": {"mem_drop_pct": float(mem_drop * 100),
                   "net_at_10pct": float(mem_drop - 0.10),
                   "net_at_20pct": float(mem_drop - 0.20),
                   "net_at_30pct": float(mem_drop - 0.30),
                   "packing_eff": {str(r[0]): float(r[2]) for r in pack_rows}},
}
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES))
print("\n".join(_LINES[-8:]))
print("\n[done] results.json + stdout.txt written")

  额外开销 30% -> 净收益 +0.003  成立
packing 效率（理论 bit/8 vs 实际对齐到 32-bit word 的占用）:
  2-bit: 每 word 装 16 个，占用 32/32 bit，packing 效率 100.0%
  3-bit: 每 word 装 10 个，占用 30/32 bit，packing 效率 93.8%
  4-bit: 每 word 装  8 个，占用 32/32 bit，packing 效率 100.0%
  6-bit: 每 word 装  5 个，占用 30/32 bit，packing 效率 93.8%
  8-bit: 每 word 装  4 个，占用 32/32 bit，packing 效率 100.0%
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/mixed_precision/results/mp_f_hardware_check.png

[done] results.json + stdout.txt written
